# Papildomas tyrimas: Trūkstamų reikšmių užpildymas (linijinė interpoliacija)
# 
# Metodas: kiekviename regione (geo), rūšiuojant pagal metus,
# trūkstama reikšmė užpildoma kaip dviejų kaimyninių reikšmių vidurkis.
# Pvz.: [12, NaN, 8] → [12, 10, 8]

In [6]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
from datetime import datetime

INPUT_CSV  = "uploads/nuts2_economic_indicators_2026-02-27.csv"
timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_CSV  = f"uploads/Interpolated_{timestamp}.csv"
OUTPUT_XLSX = f"uploads/Interpolated_{timestamp}.xlsx"

In [7]:
# Duomenų įkėlimas
df_orig = pd.read_csv(INPUT_CSV)
df_orig = df_orig.sort_values(["geo", "year"]).reset_index(drop=True)

print(f"Eilučių skaičius: {len(df_orig)}")
print(f"Stulpelių skaičius: {len(df_orig.columns)}")
print(f"Unikalių regionų skaičius: {df_orig['geo'].nunique()}")

numeric_cols = [c for c in df_orig.columns if c not in ("geo", "year")]
total_missing_before = df_orig[numeric_cols].isna().sum().sum()
print(f"\nTrūkstamų reikšmių prieš užpildymą: {total_missing_before}")

Eilučių skaičius: 6100
Stulpelių skaičius: 123
Unikalių regionų skaičius: 244

Trūkstamų reikšmių prieš užpildymą: 294633


In [8]:
# Interpoliacija: kiekviename regione atskirai, tik tarp esamų reikšmių
# limit_direction='both' + limit=1 užtikrina, kad užpildoma tik viena trūkstama
# reikšmė tarp dviejų žinomų (kaimynų vidurkis = linijinė interpoliacija).
# Kraštinės trūkstamos reikšmės (pradžioje/pabaigoje) NEUŽPILDOMOS.

df_filled = df_orig.copy()

# Saugome žymėjimą: kurios ląstelės buvo užpildytos
filled_mask = pd.DataFrame(False, index=df_orig.index, columns=df_orig.columns)

for region, grp in df_orig.groupby("geo", sort=False):
    idx = grp.index
    for col in numeric_cols:
        series = grp[col].copy()
        # Linijinė interpoliacija tik tarp esamų reikšmių (ne kraštinės)
        interpolated = series.interpolate(method="linear", limit_direction="forward", limit_area="inside")
        # Kur buvo NaN prieš, o dabar užpildyta – žymime
        was_nan  = series.isna()
        now_filled = was_nan & interpolated.notna()
        df_filled.loc[idx, col] = interpolated.values
        filled_mask.loc[idx[now_filled], col] = True

total_missing_after  = df_filled[numeric_cols].isna().sum().sum()
total_filled = filled_mask[numeric_cols].sum().sum()
print(f"Užpildyta reikšmių:            {total_filled}")
print(f"Trūkstamų reikšmių po užpildymo: {total_missing_after}")
print(f"Neužpildytos (kraštinės / izoliuotos): {total_missing_after}")

Užpildyta reikšmių:            18911
Trūkstamų reikšmių po užpildymo: 275722
Neužpildytos (kraštinės / izoliuotos): 275722


In [9]:
# CSV išsaugojimas
df_filled.to_csv(OUTPUT_CSV, index=False)
print(f"CSV išsaugotas: {OUTPUT_CSV}")

CSV išsaugotas: uploads/Interpolated_20260311_204712.csv


In [10]:
# XLSX išsaugojimas su žaliu fonu užpildytoms ląstelėms
df_filled.to_excel(OUTPUT_XLSX, index=False, engine="openpyxl")

GREEN_FILL = PatternFill(start_color="00FF00", end_color="00FF00", fill_type="solid")

wb = load_workbook(OUTPUT_XLSX)
ws = wb.active

# Stulpelių pozicijų žemėlapis (pavadinimas → Excel stulpelio numeris, 1-based)
col_positions = {col: i + 1 for i, col in enumerate(df_filled.columns)}

# df_filled.index atitinka eilutes; Excel eilutės prasideda nuo 2 (1 = antraštė)
for df_row_idx, df_col in zip(*np.where(filled_mask.values)):
    col_name = df_filled.columns[df_col]
    excel_row = df_row_idx + 2          # +2: 1 antraštė + 0-based → 1-based
    excel_col = col_positions[col_name]
    ws.cell(row=excel_row, column=excel_col).fill = GREEN_FILL

wb.save(OUTPUT_XLSX)
print(f"XLSX išsaugotas: {OUTPUT_XLSX}")
print(f"Žaliai pažymėta ląstelių: {filled_mask[numeric_cols].sum().sum()}")

XLSX išsaugotas: uploads/Interpolated_20260311_204712.xlsx
Žaliai pažymėta ląstelių: 18911
